# CertainRAG benchmark

Runs every benchmark in `evaluation/` on a single T4 GPU. Raw scores are written to `evaluation/results/` and the tables and figures to `evaluation/outputs/`.

Each run stops starting new work after `BUDGET_HOURS`, finishes cleanly and saves its output, so it never reaches the 12-hour session limit. If the summary at the end says the budget was reached, attach this version's output as an input and run the notebook again: finished examples are restored and skipped.

Requirements: GPU accelerator, internet access, and a Hugging Face read token stored as the Kaggle secret `HF_TOKEN` with the LLM-AggreFact terms accepted.

## Setup

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
!git clone -b v0.2 https://github.com/rudranshiverma/CertainRAG.git
%cd CertainRAG
!pip install -q -e ".[bench]" lettucedetect

### Resume from an earlier run

In [ ]:
import glob
import subprocess

for archive in glob.glob("/kaggle/input/*/certainrag_results.zip"):
    subprocess.run(["unzip", "-qo", archive], check=True)
    print(f"restored {archive}")

### Run helper

In [ ]:
import shlex
import sys
import time

from IPython import get_ipython

BUDGET_HOURS = 11
DEADLINE = time.time() + BUDGET_HOURS * 3600
failures, skipped = [], []


def run(command, deadline=True):
    if deadline and time.time() > DEADLINE:
        skipped.append(command)
        return
    args = [sys.executable, "-m", *shlex.split(command)] + (["--deadline", str(DEADLINE)] if deadline else [])
    get_ipython().system(shlex.join(args))
    if get_ipython().user_ns.get("_exit_code", 0) != 0:
        failures.append(command)

## Retrieval quality

Scores each RAGBench question against its documents and against documents from a distant subset, then fits Platt scaling on the training split.

In [ ]:
run("evaluation.run_retrieval")
run("evaluation.calibrate_retrieval", deadline=False)

## Faithfulness verifiers

RAGTruth and RAGBench answers, then LLM-AggreFact claims with the full test split. Add `"minicheck-deberta"` to the list to include it.

In [ ]:
for verifier in ["factcg", "minicheck-roberta"]:
    run(f"evaluation.run_faithfulness --method {verifier}")
    run(f"evaluation.run_faithfulness --method {verifier} --datasets aggrefact --limit 0 --dev-limit 300")

## Baselines

In [ ]:
for baseline in ["hhem", "lettucedetect-base", "lettucedetect-large"]:
    run(f"evaluation.run_faithfulness --method {baseline}")

## LLM judge

Serves the judge locally with Ollama and scores the RAGTruth QA test split plus 50 answers per RAGBench subset, nested inside the verifier samples.

In [ ]:
RAGBENCH = "ragbench-pubmedqa ragbench-covidqa ragbench-techqa ragbench-emanual ragbench-cuad ragbench-delucionqa"

if time.time() < DEADLINE:
    get_ipython().system("curl -fsSL https://ollama.com/install.sh | sh")
    server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    get_ipython().system("ollama pull phi4-mini")
    run(f"evaluation.run_faithfulness --method judge --datasets ragtruth-qa {RAGBENCH} --splits test --limit 50")
    server.terminate()
else:
    skipped.append("judge")

## Report

In [ ]:
from IPython.display import Image, Markdown, display

run("evaluation.report", deadline=False)

display(Markdown(open("evaluation/outputs/results.md").read()))
for figure in ["pareto.png", "risk_coverage.png"]:
    if os.path.exists(f"evaluation/outputs/{figure}"):
        display(Image(f"evaluation/outputs/{figure}"))

## Save results

In [ ]:
get_ipython().system("zip -qr /kaggle/working/certainrag_results.zip evaluation/results evaluation/outputs")

if time.time() > DEADLINE or skipped:
    print("Time budget reached. Attach this output and run again to continue:", *skipped, sep="\n  ")
if failures:
    raise RuntimeError("Failed steps:\n  " + "\n  ".join(failures))
print("All steps finished.")